In [2]:
import os
from google.colab import drive
drive.mount('/content/drive')
path = "/content/drive/MyDrive/UTP_model_training"
os.chdir(path)
print(os.getcwd())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/UTP_model_training


In [ ]:
'''
%cd /content/drive/MyDrive/UTP_model_training
!git clone https://github.com/hiyouga/LLaMA-Factory.git
'''
%cd /content/drive/MyDrive/UTP_model_training/LLaMA-Factory/
!pip install -e .[torch,metrics]
!pip install transformers==4.45.2 tokenizers==0.20.1

In [4]:
!pip show llamafactory
!pip show bitsandbytes

import transformers,torch

print("当前 torch 的真实版本:", torch.__version__)
print("当前 transformers 的真实版本:", transformers.__version__)

Name: llamafactory
Version: 0.8.3.dev0
Summary: Easy-to-use LLM fine-tuning framework
Home-page: https://github.com/hiyouga/LLaMA-Factory
Author: hiyouga
Author-email: hiyouga@buaa.edu.cn
License: Apache 2.0 License
Location: /usr/local/lib/python3.12/dist-packages
Editable project location: /content/drive/MyDrive/UTP_model_training/LLaMA-Factory
Requires: accelerate, datasets, einops, fastapi, fire, gradio, matplotlib, numpy, packaging, pandas, peft, protobuf, pydantic, pyyaml, scipy, sentencepiece, sse-starlette, tiktoken, transformers, trl, uvicorn
Required-by: 
Name: bitsandbytes
Version: 0.49.2
Summary: k-bit optimizers and matrix multiplication routines.
Home-page: https://github.com/bitsandbytes-foundation/bitsandbytes
Author: 
Author-email: Tim Dettmers <dettmers@cs.washington.edu>
License: 
Location: /usr/local/lib/python3.12/dist-packages
Requires: numpy, packaging, torch
Required-by: 
当前 torch 的真实版本: 2.10.0+cu128
当前 transformers 的真实版本: 4.45.2


### Check GPU environment

In [2]:
import torch
try:
  assert torch.cuda.is_available() is True
except AssertionError:
  print("Please set up a GPU before using LLaMA Factory: https://medium.com/mlearning-ai/training-yolov4-on-google-colab-316f8fff99c6")

## Fine-tune model via Command Line

It takes ~30min for training.

In [ ]:
import json

args = dict(
  stage="sft",                        # do supervised fine-tuning
  do_train=True,
  #model_name_or_path="unsloth/llama-3-8b-Instruct-bnb-4bit", # use bnb-4bit-quantized Llama-3-8B-Instruct model #Meta-Llama-3.1-8B-Instruct-bnb-4bit
  model_name_or_path = "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
  dataset="UILinkII",             # use alpaca and identity datasets
  template="llama3",                     # use llama3 prompt template
  finetuning_type="lora",                   # use LoRA adapters to save memory
  lora_target="all",                     # attach LoRA adapters to all linear layers
  output_dir="finetuned_llama31",                  # the path to save LoRA adapters
  per_device_train_batch_size=2,               # the batch size
  gradient_accumulation_steps=4,               # the gradient accumulation steps
  lr_scheduler_type="cosine",                 # use cosine learning rate scheduler
  logging_steps=10,                      # log every 10 steps
  warmup_ratio=0.1,                      # use warmup scheduler
  save_steps=1000,                      # save checkpoint every 1000 steps
  learning_rate=5e-5,                     # the learning rate
  num_train_epochs=3.0,                    # the epochs of training
  max_samples=500,                      # use 500 examples in each dataset
  max_grad_norm=1.0,                     # clip gradient norm to 1.0
  quantization_bit=4,                     # use 4-bit QLoRA
  loraplus_lr_ratio=16.0,                   # use LoRA+ algorithm with lambda=16.0
  fp16=True,                         # use float16 mixed precision training
)

json.dump(args, open("train_llama3.json", "w", encoding="utf-8"), indent=2)

%cd /content/drive/MyDrive/UTP_model_training/LLaMA-Factory/

!llamafactory-cli train train_llama3.json

#Infer finetuned models

In [9]:
#from llmtuner.chat import ChatModel
#from llmtuner.extras.misc import torch_gc
from llamafactory.chat import ChatModel
from llamafactory.extras.misc import torch_gc
import json
import pandas as pd
%cd /content/drive/MyDrive/UTP_model_training/LLaMA-Factory/

#with open("data/UIT.json", "w", encoding="utf-8") as f:
#    json.dump(UIT, f, indent=2, ensure_ascii=False)

args = dict(
    #model_name_or_path="unsloth/llama-3-8b-Instruct-bnb-4bit", # use bnb-4bit-quantized Llama-3-8B-Instruct model
    model_name_or_path="unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
    adapter_name_or_path="llama3.1_UILink_II_lora_epoch6",            # load the saved LoRA adapters
    template="llama3",                     # same to the one in training
    finetuning_type="lora",                  # same to the one in training
    quantization_bit=4,                    # load 4-bit quantized model
)
chat_model = ChatModel(args)

with open("data/UILinkII_test.json", "r", encoding="utf-8") as f:
    dataset = json.load(f)

correct = 0
pair_num = 0
result_dict = {
        'pair_num': [],
        'result': [],
        'human_label': []
    }

for sample in dataset:
    pair_num+=1
    messages = []
    query = sample["instruction"] +'\n'+ sample["input"]
    messages.append({"role": "user", "content": query})

    print("Assistant: ", end="", flush=True)

    response = ""
    for new_text in chat_model.stream_chat(messages):
        print(new_text, end="", flush=True)
        response += new_text
    print('\n')
    print("ground truth:", sample["output"])

    result_dict['pair_num'].append(pair_num)
    result_dict['result'].append(response)
    result_dict['human_label'].append(sample["output"])


    df = pd.DataFrame(result_dict)
    df.to_csv("json_result.csv", index=False)



[INFO|tokenization_utils_base.py:2206] 2026-05-17 13:40:14,044 >> loading file tokenizer.json from cache at /root/.cache/huggingface/hub/models--unsloth--Meta-Llama-3.1-8B-Instruct-bnb-4bit/snapshots/f15c379fb32bb402fa06a7ae9aecb1febf4b79ec/tokenizer.json
[INFO|tokenization_utils_base.py:2206] 2026-05-17 13:40:14,045 >> loading file tokenizer.model from cache at None
[INFO|tokenization_utils_base.py:2206] 2026-05-17 13:40:14,046 >> loading file added_tokens.json from cache at None
[INFO|tokenization_utils_base.py:2206] 2026-05-17 13:40:14,046 >> loading file special_tokens_map.json from cache at /root/.cache/huggingface/hub/models--unsloth--Meta-Llama-3.1-8B-Instruct-bnb-4bit/snapshots/f15c379fb32bb402fa06a7ae9aecb1febf4b79ec/special_tokens_map.json
[INFO|tokenization_utils_base.py:2206] 2026-05-17 13:40:14,047 >> loading file tokenizer_config.json from cache at /root/.cache/huggingface/hub/models--unsloth--Meta-Llama-3.1-8B-Instruct-bnb-4bit/snapshots/f15c379fb32bb402fa06a7ae9aecb1feb

/content/drive/MyDrive/UTP_model_training/LLaMA-Factory


[INFO|tokenization_utils_base.py:2470] 2026-05-17 13:40:14,793 >> Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


05/17/2026 13:40:14 - INFO - llamafactory.data.template - Replace eos token: <|eot_id|>


INFO:llamafactory.data.template:Replace eos token: <|eot_id|>
[INFO|configuration_utils.py:675] 2026-05-17 13:40:14,915 >> loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--unsloth--Meta-Llama-3.1-8B-Instruct-bnb-4bit/snapshots/f15c379fb32bb402fa06a7ae9aecb1febf4b79ec/config.json
[INFO|configuration_utils.py:742] 2026-05-17 13:40:14,917 >> Model config LlamaConfig {
  "_name_or_path": "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
  "architectures": [
    "LlamaForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 128000,
  "eos_token_id": 128009,
  "head_dim": 128,
  "hidden_act": "silu",
  "hidden_size": 4096,
  "initializer_range": 0.02,
  "intermediate_size": 14336,
  "max_position_embeddings": 131072,
  "mlp_bias": false,
  "model_type": "llama",
  "num_attention_heads": 32,
  "num_hidden_layers": 32,
  "num_key_value_heads": 8,
  "pad_token_id": 128004,
  "pretraining_tp": 1,
  "quantization_config": {
 

...

05/17/2026 13:40:29 - INFO - llamafactory.model.adapter - Loaded adapter(s): llama3.1_UILink_lora_epoch6


INFO:llamafactory.model.adapter:Loaded adapter(s): llama3.1_UILink_lora_epoch6


05/17/2026 13:40:29 - INFO - llamafactory.model.loader - all params: 8,051,232,768


INFO:llamafactory.model.loader:all params: 8,051,232,768


Assistant: {'id': 9, reason': 'Click to share the content on Facebook\n'}

ground truth: {'id': 4, 'reason': ''}
Assistant: {'id': 4, reason': 'it should be the go back \n'}

ground truth: {'id': 4, 'reason': ''}
Assistant: {'id': 5, reason': 'user clicked "proceed" for the next step\n'}

ground truth: {'id': 5, 'reason': ''}
Assistant: {'id': 1, reason': 'the chosen element means next\n'}

ground truth: {'id': 1, 'reason': ''}
Assistant: {'id': 1, reason': 'the chosen element means back to the main page\n'}

ground truth: {'id': 3, 'reason': ''}
Assistant: {'id': 1, reason': 'because it takes us to the next page\n'}

ground truth: {'id': 1, 'reason': ''}
Assistant: {'id': 1, reason': 'the chosen element means back to the main page\n'}

ground truth: {'id': 4, 'reason': ''}
Assistant: {'id': 4, reason': 'click the menu button to show the menu\n'}

ground truth: {'id': 4, 'reason': ''}
Assistant: {'id': 1, reason': 'the chosen element means back to the main page\n'}

ground truth: {'id'

#Experiments using different prompts to llama3.1


In [12]:
import re
from abc import abstractmethod
from typing import List
from http import HTTPStatus
import base64
import requests
import os
import json
import cv2 as cv
import pandas as pd

from llmtuner.chat import ChatModel
from llmtuner.extras.misc import torch_gc

%cd /content/drive/MyDrive/UTP_model_training/LLaMA-Factory/

#with open("data/UIT.json", "w", encoding="utf-8") as f:
#    json.dump(UIT, f, indent=2, ensure_ascii=False)


with open("data/UILinkII_test.json", "r", encoding="utf-8") as f:
    testset = json.load(f)

print(testset[43])
print(testset[88])
print(testset[95])
print(testset[112])

system_prompt = "You are an ordinary smartphone user who can understand the transition logic between consecutive GUI screens. You will be given a pair of consecutive smartphone GUI screens, you need to identify the id of the UI element on the prior screen that link the prior UI screen to the next UI screen. If you fail to do this, also explain your reason."
system_knowledge_prompt = "You can reason the navigation relationship with the following 5 principles: \n Principle (1). Comparing semantic consistency, and choose the UI element that is related with the main topic of the next screen. \n Principle (2). Reasoning the logic or workflow and the hierarchical relationship beteen the two screens (e.g., go back to the home screen). \n Principle (3). Comparing the visual variant between two GUI screens and choose the UI element that got larger or highlighted. \n Principle (4). Understand and recognize the common navigation mode. \n Principle (5). Choosing the most visually salient UI element."

user_prompt = "Please describe how to transit from the prior UI screen to the later UI screen. You need to identify the index of the UI element that link the prior UI screen to the next UI screen, and explain your reason for such a choice. If you feel they are not consecutive screens or have no link UI, also explain your reason."
example_pairs = [testset[88],testset[95],testset[112],testset[43]]

example_without_knowledge_response = [
    "id:8. On the target screen, the icon \'Plans\' located in the bottom nav bar is highlighted \n The link UI element is the \'Plans\' icon, the id is 8",
    "id:2. The target screen appears to be displaying a passage from the Bible, which suggests clicking on the call-to-action on the first screen. \n The link UI element is the \'call-to-action\' button in the middle screen, the id is 2",
    "id:None. The prior screen and the next screen are of large visual difference. \n No link UI",
    "id:1. The target screen is the home page so I will tap the \'back\' icon to get back. \n The link UI element is the \'back\' icon, the id is 1"]
    #"id:2. The target screen list many online shops, so I will tap the \'Purchases\' icon to transit to that screen. \n The link UI element is the \'Purchases\' item, the id is No.2"]


example_COT_knowledge_response = [
    "id:8. According to Principle (3). Comparing the visual variant between two GUI screens and choose the UI element that got larger. \n On the second screen the icon \'Plans\' located in the bottom nav bar is highlighted, while on the first screen the highligted icon is the \'Read \'. This means the user tapped the UI element \'Plan\' on the first screen so that the highlighted icon changed from  \'Plan\' to \'Read\'. \n The link UI element is the \'Plan\' icon. The id number is 8",
    "id:2.According to Principle (1). Comparing semantic consistency and choose the UI element that is related with the main topic of the next screen. According to principle (5). Choosing the most visually salient UI element. \n The second screen appears to be displaying a passage from the Bible, which suggests that clicking on the corresponding call-to-action on the first screen. The arrow icon on the center of the first screen is very visually salient and attract users' attention with the meaning of navigating to the next screen of the Bible content. \n The link UI element is the arrow icon on the center of the first screen. The id number is 2",
    "id:None. No applicable principles works for this case. \n The prior screen and the next screen are of large visual difference so that it is hard to reason the transition logic. \n No link UI",
    "id:1. According to Principle (2). Reasoning the logic or workflow and the hierarchical relationship beteen the two screens (e.g., go back to the home screen). \n The target screen is the home page so I will tap the \'back\' icon to get back. \n The link UI element is the \'back\' icon. The id is 1"]
    #"According to Principle (1). Comparing semantic consistency and choose the UI element that is related with the main topic of the next screen. \n The target screen list many online shops, so I will tap the \'Purchases\' icon to transit to that screen. \n The link UI element is the \'Purchases\' item.  The index number is No.2"]

class LinkModel():
    def __init__(self):
        args = dict(
            #model_name_or_path="unsloth/llama-3-8b-Instruct-bnb-4bit", # use bnb-4bit-quantized Llama-3-8B-Instruct model
            model_name_or_path="unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
            adapter_name_or_path="llama3.1_UILink_II_lora_epoch6",            # load the saved LoRA adapters
            template="llama3",                     # same to the one in training
            finetuning_type="lora",                  # same to the one in training
            quantization_bit=4,                    # load 4-bit quantized model
            temperature = 0.001
        )
        self.chat_model = ChatModel(args)


    def few_shot_prompot_llama31_response(self, prompt):
        shot1 = example_pairs[0]["instruction"] + "\n" + example_pairs[0]["input"]
        response1 = example_without_knowledge_response[0]

        shot2 = example_pairs[1]["instruction"] + "\n" + example_pairs[1]["input"]
        response2 = example_without_knowledge_response[1]

        shot3 = example_pairs[2]["instruction"] + "\n" + example_pairs[2]["input"]
        response3 = example_without_knowledge_response[2]

        shot4 = example_pairs[3]["instruction"] + "\n" + example_pairs[3]["input"]
        response4 = example_without_knowledge_response[3]

        messages = []

        messages.append({"role": "user","content": shot1})
        messages.append({"role": "assistant","content": response1})
        messages.append({"role": "user","content": shot2})
        messages.append({"role": "assistant","content": response2})
        messages.append({"role": "user","content": shot3})
        messages.append({"role": "assistant","content": response3})
        messages.append({"role": "user","content": shot4})
        messages.append({"role": "assistant","content": response4})
        messages.append({"role": "user", "content": prompt})

        print("Assistant: ", end="", flush=True)
        response = ""
        for new_text in self.chat_model.stream_chat(messages):
            print(new_text, end="", flush=True)
            response += new_text
        print('\n')
        #print("ground truth:", sample["output"])
        return True, response

    def few_shot_prompt_human_knowledge_COT_llama31(self, prompt):

        shot1 = example_pairs[0]["instruction"] + "\n" + example_pairs[0]["input"]
        response1 = example_COT_knowledge_response[0]

        shot2 = example_pairs[1]["instruction"] + "\n" + example_pairs[1]["input"]
        response2 = example_COT_knowledge_response[1]

        shot3 = example_pairs[2]["instruction"] + "\n" + example_pairs[2]["input"]
        response3 = example_COT_knowledge_response[2]

        shot4 = example_pairs[3]["instruction"] + "\n" + example_pairs[3]["input"]
        response4 = example_COT_knowledge_response[3]

        messages = []

        messages.append({"role": "user","content": shot1})
        messages.append({"role": "assistant","content": response1})
        messages.append({"role": "user","content": shot2})
        messages.append({"role": "assistant","content": response2})
        messages.append({"role": "user","content": shot3})
        messages.append({"role": "assistant","content": response3})
        messages.append({"role": "user","content": shot4})
        messages.append({"role": "assistant","content": response4})
        #messages.append({"role": "user", "content": system_knowledge_prompt})
        messages.append({"role": "user", "content": prompt})

        print("Assistant: ", end="", flush=True)
        response = ""
        for new_text in self.chat_model.stream_chat(messages):
            print(new_text, end="", flush=True)
            response += new_text
        print('\n')
        #print("ground truth:", sample["output"])
        return True, response

    def get_model_response_llama31_with_human_knowledge(self, prompt):
        messages = []
        #messages.append({"role": "user", "content": system_knowledge_prompt})
        messages.append({"role": "user", "content": system_knowledge_prompt + '\n'+ prompt})
        print("Assistant: ", end="", flush=True)

        response = ""
        for new_text in self.chat_model.stream_chat(messages):
            print(new_text, end="", flush=True)
            response += new_text
        print('\n')
        #print("ground truth:", sample["output"])
        return True, response

    def get_model_response_llama31(self, prompt):
        messages = []
        messages.append({"role": "user", "content": prompt})
        print("Assistant: ", end="", flush=True)

        response = ""
        for new_text in self.chat_model.stream_chat(messages):
            print(new_text, end="", flush=True)
            response += new_text
        print('\n')
        #print("ground truth:", sample["output"])
        return True, response

def promptllama31(mode, dataset):
    result_dict = {
        'pair_num': [],
        'result': [],
        'human_label': []
    }

    for i in range(1, 120):
        if i == 44 or i==89 or i==96 or i==113:
            continue
        prompt = testset[i-1]["instruction"] + "\n" + testset[i-1]["input"]

        rsp = None
        if mode == "direct_prompt":
            status, rsp = mllm.get_model_response_llama31(prompt)
        if mode == "prompt_with_knowledge":
            status, rsp = mllm.get_model_response_llama31_with_human_knowledge(prompt)
        if mode == "few_shot_without_knowledge":
            status, rsp = mllm.few_shot_prompot_llama31_response(prompt)
        if mode == "few_shot_with_COT_knowledge":
            status, rsp = mllm.few_shot_prompt_human_knowledge_COT_llama31(prompt)

        print(i, rsp)
        result_dict['pair_num'].append(i)
        result_dict['result'].append(rsp)
        result_dict['human_label'].append(testset[i-1]["output"])


        df = pd.DataFrame(result_dict)
        df.to_csv(mode + "_"+dataset+"jsonII_llama31_UILinkII_loracheck.csv", index=False)


mllm = LinkModel()
modes = ["direct_prompt", "prompt_with_knowledge", "few_shot_without_knowledge", "few_shot_with_COT_knowledge"]
promptllama31(modes[1], "test")





[INFO|tokenization_utils_base.py:2206] 2026-05-17 14:57:32,551 >> loading file tokenizer.json from cache at /root/.cache/huggingface/hub/models--unsloth--Meta-Llama-3.1-8B-Instruct-bnb-4bit/snapshots/f15c379fb32bb402fa06a7ae9aecb1febf4b79ec/tokenizer.json
[INFO|tokenization_utils_base.py:2206] 2026-05-17 14:57:32,552 >> loading file tokenizer.model from cache at None
[INFO|tokenization_utils_base.py:2206] 2026-05-17 14:57:32,552 >> loading file added_tokens.json from cache at None
[INFO|tokenization_utils_base.py:2206] 2026-05-17 14:57:32,553 >> loading file special_tokens_map.json from cache at /root/.cache/huggingface/hub/models--unsloth--Meta-Llama-3.1-8B-Instruct-bnb-4bit/snapshots/f15c379fb32bb402fa06a7ae9aecb1febf4b79ec/special_tokens_map.json
[INFO|tokenization_utils_base.py:2206] 2026-05-17 14:57:32,553 >> loading file tokenizer_config.json from cache at /root/.cache/huggingface/hub/models--unsloth--Meta-Llama-3.1-8B-Instruct-bnb-4bit/snapshots/f15c379fb32bb402fa06a7ae9aecb1feb

/content/drive/MyDrive/UTP_model_training/LLaMA-Factory
{'instruction': 'You are a smartphone user, and plan to transit from the current UI screen to the target UI screen, please identify the id of the UI element on the preceding screen which can transit to the suceeding screen.', 'input': '{\'current UI screen\': [{\'left\': 6, \'top\': 30, \'right\': 66, \'down\': 72, \'text\': \'Left arrow icon, typically used for navigating back to the previous screen or page.\', \'id\': 1}, {\'left\': 372, \'top\': 30, \'right\': 429, \'down\': 73, \'text\': \'Button with the text "Next" in blue, indicating an action to proceed to the next step or page.\', \'id\': 2}, {\'left\': 110, \'top\': 378, \'right\': 337, \'down\': 534, \'text\': \'"Difficulty selection interface with three options: \\\'Easy\\\' (grayed out), \\\'Average\\\' (highlighted in blue with a checkmark), and \\\'Intense\\\' (grayed out)."\', \'id\': 3}, {\'left\': 69, \'top\': 721, \'right\': 130, \'down\': 763, \'text\': \'"Back

[INFO|tokenization_utils_base.py:2470] 2026-05-17 14:57:33,357 >> Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


05/17/2026 14:57:33 - INFO - llmtuner.data.template - Replace eos token: <|eot_id|>


INFO:llmtuner.data.template:Replace eos token: <|eot_id|>
[INFO|configuration_utils.py:675] 2026-05-17 14:57:33,476 >> loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--unsloth--Meta-Llama-3.1-8B-Instruct-bnb-4bit/snapshots/f15c379fb32bb402fa06a7ae9aecb1febf4b79ec/config.json
[INFO|configuration_utils.py:742] 2026-05-17 14:57:33,477 >> Model config LlamaConfig {
  "_name_or_path": "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
  "architectures": [
    "LlamaForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 128000,
  "eos_token_id": 128009,
  "head_dim": 128,
  "hidden_act": "silu",
  "hidden_size": 4096,
  "initializer_range": 0.02,
  "intermediate_size": 14336,
  "max_position_embeddings": 131072,
  "mlp_bias": false,
  "model_type": "llama",
  "num_attention_heads": 32,
  "num_hidden_layers": 32,
  "num_key_value_heads": 8,
  "pad_token_id": 128004,
  "pretraining_tp": 1,
  "quantization_config": {
    "

...

05/17/2026 14:57:37 - INFO - llmtuner.model.adapter - Loaded adapter(s): llama3.1_UILink_II_lora_epoch6


INFO:llmtuner.model.adapter:Loaded adapter(s): llama3.1_UILink_II_lora_epoch6


05/17/2026 14:57:37 - INFO - llmtuner.model.loader - all params: 8051232768


INFO:llmtuner.model.loader:all params: 8051232768


Assistant: {'id': 4, reason': 'from the next page we can get it\n'}

1 {'id': 4, reason': 'from the next page we can get it\n'}
Assistant: {'id': 2, reason': 'prepare to take the picture\n'}

2 {'id': 2, reason': 'prepare to take the picture\n'}
Assistant: {'id': 5, reason': 'touch here to choose an option\n'}

3 {'id': 5, reason': 'touch here to choose an option\n'}
Assistant: {'id': 1, reason': 'use the icon to select the option\n'}

4 {'id': 1, reason': 'use the icon to select the option\n'}
Assistant: {'id': 2, reason': 'the chosen element means menu\n'}

5 {'id': 2, reason': 'the chosen element means menu\n'}
Assistant: {'id': 1, reason': 'the content shows up again\n'}

6 {'id': 1, reason': 'the content shows up again\n'}
Assistant: {'id': 3, reason': 'the ui element means take the test\n'}

7 {'id': 3, reason': 'the ui element means take the test\n'}
Assistant: {'id': 4, reason': 'this ui element means show the setting of the app\n'}

8 {'id': 4, reason': 'this ui element means 